In [1]:
using Pkg
Pkg.activate("/Users/jkarpowski/Dendron/notes/assets/scripts/")
using SparseArrays
using LinearAlgebra
using BenchmarkTools

using Revise
using Arrhenius

  Activating project at `~/Dendron/notes/assets/scripts`


# Type definition

In [3]:
struct UnifiedReactions{Ty<:AbstractArray,sTy<:AbstractArray,vTy<:AbstractVector}
    arrhenius_coeffs::Ty
    reactant_orders::sTy
    product_stoich_coeffs::sTy
    reactant_stoich_coeffs::sTy
    vk::sTy
    vk_rev::sTy
    vk_sum::vTy
    efficiency_coeffs_tB::sTy
    efficiency_coeffs_fO::sTy
    Arrhenius_0::Ty
    troe::Ty
    index_rev_start::Int64
    index_three_body_start::Int64
    index_fallOff_start::Int64
    index_troe_start::Int64
end

mutable struct UnifiedState{vTy<:AbstractVector}
    kf::vTy
    kr::vTy
    C::vTy
    s0_R::vTy
    h_RT::vTy
    g_RT::vTy
end

# generation of random ref Point

In [27]:
gas = CreateSolution("../mechanism/h2_12s.yaml",sparseCoeffs=true);

X = rand(gas.n_species)
X = X ./ sum(X);
Y = X2Y(gas, X, dot(X, gas.MW));
P = 1e5
T = 900

mean_MW = 1.0 / dot(Y, 1 ./ gas.MW)
ρ_mass = P / R / T * mean_MW
C = Y2C(gas, Y, ρ_mass)
h_RT = Arrhenius.cal_h_RT(gas, T, P, X)
s0_R = Arrhenius.cal_s0_R(gas, T, P, X)
g_RT = h_RT - s0_R

kr = rand(gas.n_reactions);
kf = zeros(gas.n_reactions);
wdot = zeros(gas.n_species)

unified_state= UnifiedState(
    zeros(gas.n_reactions),
    zeros(gas.n_reactions),
    C,
    s0_R,
    h_RT,
    g_RT
)

UnifiedState{Vector{Float64}}([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0027230949678073383, 0.0010460415364446757, 0.0019041463634760938, 0.0012442674315776066, 0.00048460288486219767, 0.002381958518177534, 0.0009369787781408075, 0.001092207453037273, 0.001550297071223811], [19.61222768540597, 28.858031474835983, 27.476822373649302, 16.559304055544153, 22.20019985026545, 26.04155337454115, 32.89908821804394, 34.98282303805138, 27.021457568348346], [2.3622939778658, 2.5717640261477888, -29.384606218660927, 30.804066555631643, 35.00464980388398, 7.407424052334077, 4.999953565951689, -13.920339589113066, 2.4353343846477777], [-17.24993370754017, -26.286267448688193, -56.86142859231023, 14.24476250008749, 12.80444995361853, -18.634129322207073, -27.899134652092254, -48.90316262716445, -24.58612318370057])

# Generate kinetics and Reactions

In [28]:
i_forward = .!(gas.kinetics.is_reversible)
i_rev = Int64[]
for (i) in 1:gas.n_reactions
    if !(i in  gas.kinetics.index_three_body) & !(i in  gas.kinetics.index_falloff) & gas.kinetics.is_reversible[i]
        push!(i_rev,i)
    end

end
i_tB = gas.kinetics.index_three_body
i_fO = Int64[]
i_troe = Int64[]
i_A0_fO = Int64[]
i_A0_troe = Int64[]
for (i, index) in enumerate(gas.kinetics.index_falloff )
    if gas.kinetics.index_falloff_Troe[i] < 0
        push!(i_fO,index)
        push!(i_A0_fO,i)
    else
        push!(i_troe,index)
        push!(i_A0_troe,i)
    end
end

arr_coeffs = vcat(
    gas.kinetics.Arrhenius_coeffs[i_forward,:],
    gas.kinetics.Arrhenius_coeffs[i_rev,:],
    gas.kinetics.Arrhenius_coeffs[i_tB,:],
    gas.kinetics.Arrhenius_coeffs[i_fO,:],
    gas.kinetics.Arrhenius_coeffs[i_troe,:],
    )

reactant_orders = hcat(
    gas.kinetics.reactant_orders[:,i_forward],
    gas.kinetics.reactant_orders[:,i_rev],
    gas.kinetics.reactant_orders[:,i_tB],
    gas.kinetics.reactant_orders[:,i_fO],
    gas.kinetics.reactant_orders[:,i_troe])

product_stoich_coeffs = hcat(
    gas.kinetics.product_stoich_coeffs[:,i_rev],
    gas.kinetics.product_stoich_coeffs[:,i_tB],
    gas.kinetics.product_stoich_coeffs[:,i_fO],
    gas.kinetics.product_stoich_coeffs[:,i_troe])

reactant_stoich_coeffs = hcat(
    gas.kinetics.reactant_stoich_coeffs[:,i_rev],
    gas.kinetics.reactant_stoich_coeffs[:,i_tB],
    gas.kinetics.reactant_stoich_coeffs[:,i_fO],
    gas.kinetics.reactant_stoich_coeffs[:,i_troe])

vk = sparse(vcat(
    gas.kinetics.vk[:,i_forward]',
    gas.kinetics.vk[:,i_rev]',
    gas.kinetics.vk[:,i_tB]',
    gas.kinetics.vk[:,i_fO]',
    gas.kinetics.vk[:,i_troe]'
    ))

vk_rev = sparse(vcat(
    gas.kinetics.vk[:,i_rev]',
    gas.kinetics.vk[:,i_tB]',
    gas.kinetics.vk[:,i_fO]',
    gas.kinetics.vk[:,i_troe]'))

vk_sum = vcat(
    gas.kinetics.vk_sum[i_rev],
    gas.kinetics.vk_sum[i_tB],
    gas.kinetics.vk_sum[i_fO],
    gas.kinetics.vk_sum[i_troe]
)

efficiency_coeffs_tB=gas.kinetics.efficiencies_coeffs[:,i_tB]

efficiency_coeffs_fO = hcat(
    gas.kinetics.efficiencies_coeffs[:,i_fO],
    gas.kinetics.efficiencies_coeffs[:,i_troe])

Arrhenius_0 =vcat(
    gas.kinetics.Arrhenius_0[i_A0_fO,:],
    gas.kinetics.Arrhenius_0[i_A0_troe,:])

troe =gas.kinetics.Troe_

index_rev_start = sum(i_forward)+1
index_three_body_start = index_rev_start + length(i_rev) 
index_fallOff_start = index_three_body_start + length(i_tB) 
index_troe_start = index_fallOff_start + length(i_fO) 
index_troe_start+length(i_troe)

unifiedReactions = UnifiedReactions(
    arr_coeffs,
    reactant_orders,
    product_stoich_coeffs,
    reactant_stoich_coeffs,
    vk,
    vk_rev,
    vk_sum,
    efficiency_coeffs_tB,
    efficiency_coeffs_fO,
    Arrhenius_0,
    troe,
    index_rev_start,
    index_three_body_start,
    index_fallOff_start,
    index_troe_start,
)




UnifiedReactions{Matrix{Float64}, SparseMatrixCSC{Float64, Int64}, Vector{Float64}}([7.08e10 0.0 295.0; 2.89e10 0.0 -497.13; … ; 4.65e9 0.44 0.0; 2.62328e19 -1.38836 51323.0], sparse([4, 7, 6, 7, 7, 1, 7, 2, 4, 1  …  1, 6, 4, 7, 4, 6, 4, 2, 4, 8], [1, 1, 2, 2, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 11, 11, 12], [1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0  …  1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 1.0], 9, 12), sparse([5, 6, 4, 6, 3, 4, 1, 2, 3, 1, 7, 6], [1, 1, 2, 2, 3, 3, 4, 4, 5, 6, 7, 8], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0], 9, 8), sparse([2, 4, 1, 5, 1, 6, 4, 7, 4, 6, 4, 2, 4, 8], [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 7, 7, 8], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0, 1.0, 1.0, 1.0], 9, 8), sparse([4, 6, 7, 8, 10, 2, 3, 5, 8, 11  …  12, 1, 2, 3, 4, 8, 11, 3, 4, 12], [1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 8, 8, 8], [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 

# kernels

In [22]:
#forwardRate
function forwardRate!(kf::AbstractVector, arr_coeffs::AbstractArray,T::Real)
    logT = log(T)
    rec_RcalT = (4184.0 / Arrhenius.R / T)
    forwardRate!(kf,arr_coeffs,logT,rec_RcalT)
    return nothing
end
function forwardRate!(kf::AbstractVector, arr_coeffs::AbstractArray,logT::Real,rec_RcalT::Real)
    for i in eachindex(kf) 
        @inbounds  kf[i] = arr_coeffs[i, 1]  * exp(arr_coeffs[i, 2] * logT - arr_coeffs[i, 3] * rec_RcalT)
    end
    return nothing
end

function kf_Pi_c_rectantOrder!(kf,C,reactant_orders)
    for i in eachindex(kf)
        for j in reactant_orders.colptr[i]:reactant_orders.colptr[i+1]-1
            @inbounds kf[i] *= C[reactant_orders.rowval[j]]^reactant_orders.nzval[j]
        end
    end
end

function kr!(kr, kf,g_RT,vk,vk_sum,T)
    mul!(kr,vk,g_RT) #kr proxies as Δg_RT as intermediate storage'
    logP = log(one_atm / R / T)
    @. kr = kf / exp(-kr + logP * vk_sum)
    return nothing
end

function vk_kf!(wdot,kf,vk) # TODO use likwid to compare the two methods
    # transposed vk - workflow 
    for isp in eachindex(wdot) 
        for j in vk.colptr[isp]:vk.colptr[isp+1]-1
            wdot[isp] += vk.nzval[j] * kf[vk.rowval[j]]
        end
    end 
    # for ir in eachindex(kf)
    #     for j in vk.colptr[ir]:vk.colptr[ir+1]-1
    #         wdot[vk.rowval[j]] += vk.nzval[j] * kf[ir]
    #     end
    # end
    #mul!(wdot,vk',kf)
    return nothing
end


function efficency!( kf,effCoeff,C)
    for ir in eachindex(kf)
        dot_C_eff = 0.0
        @inbounds for j in effCoeff.colptr[ir]:effCoeff.colptr[ir+1]-1
            @inbounds dot_C_eff += effCoeff.nzval[j] * C[effCoeff.rowval[j]]
        end
        @inbounds kf[ir] *= dot_C_eff
    end
    return nothing
end

function troe!(kf, logPr,troeCoeffs,T)
    for i in eachindex(kf)
      @inbounds F_cent =
                  (1 - troeCoeffs[i, 1]) * exp(-T / troeCoeffs[i, 4]) +
                  troeCoeffs[i, 1] * exp(-T / troeCoeffs[i, 2]) +
                  exp(-troeCoeffs[i, 3] / T)
      lF_cent = log10(F_cent)
      _C = -0.4 - 0.67 * lF_cent
      N = 0.75 - 1.27 * lF_cent
      f1 = (logPr[i] + _C) / (N - 0.14 * (logPr[i] + _C))
      @inbounds kf[i] *= exp(log(10.0) * lF_cent / (1 + f1^2))
    end
  end

troe! (generic function with 1 method)

# update Functions and tests 

In [29]:
function wdot_func!(wdot::AbstractVector,kf::AbstractVector,kr::AbstractVector,C::AbstractVector,g_RT::AbstractVector,reactions::UnifiedReactions ,T::Real)
    wdot.=0.0
    logT = log(T)
    rec_RcalT = (4184.0 / Arrhenius.R / T)
    forwardRate!(kf, reactions.arrhenius_coeffs,logT, rec_RcalT)

    #threeBody
    tB_range = reactions.index_three_body_start:reactions.index_fallOff_start-1
    efficency!(@view(kf[tB_range]),reactions.efficiency_coeffs_tB,C)

    #fallOff adaptation
    fO_start=  reactions.index_fallOff_start#:end#reactions.index_fallOff_start-1
    forwardRate!(@view(kr[fO_start:end]),reactions.Arrhenius_0,logT, rec_RcalT) #kr proxies as k0
    efficency!(@view(kr[fO_start:end]),reactions.efficiency_coeffs_fO,C) #kr proxies as k0 * dot(eff,C)
    @inbounds @view(kr[fO_start:end]) ./= @view(kf[fO_start:end]) #kr proxies as Pr
    @inbounds @. @view(kf[fO_start:end]) *= (@view(kr[fO_start:end]) / (1 + @view(kr[fO_start:end])))

    #troe
    troe_start = reactions.index_troe_start
    @inbounds kr[troe_start:end] .= log10.(@view(kr[troe_start:end])) #kr proxies as logPr
    troe!(@view(kf[troe_start:end]),@view(kr[troe_start:end]), reactions.troe,T)


    kr!(@view(kr[reactions.index_rev_start:end]),@view(kf[reactions.index_rev_start:end]),g_RT,reactions.vk_rev,reactions.vk_sum,T)
    kf_Pi_c_rectantOrder!(kf,C,reactions.reactant_orders)
    kf_Pi_c_rectantOrder!(@view(kr[reactions.index_rev_start:end]),C,reactions.product_stoich_coeffs)
    kf .-=kr
    vk_kf!(wdot,kf,reactions.vk)
    return nothing
end
@benchmark wdot_func!($wdot,$unified_state.kf,$unified_state.kr,$unified_state.C,$unified_state.g_RT,$unifiedReactions,$T) evals=20

BenchmarkTools.Trial: 10000 samples with 20 evaluations.
 Range (min … max):  745.350 ns …  24.297 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     976.000 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):     1.203 μs ± 758.518 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █▄▂▂▁▁▁▂▄▅▅▄▃▃▃▂▁▁▁▁▁▁▁▁▁                                     ▁
  █████████████████████████████▇▇█▇▇▇▇▇▆▇▆▆▇▆▆▆▅▄▆▅▆▅▅▆▅▄▅▅▃▅▃▅ █
  745 ns        Histogram: log(frequency) by time       3.81 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [31]:
@benchmark wdot_old($gas.kinetics, $T, $C,$s0_R,$h_RT) 

BenchmarkTools.Trial: 100000 samples with 10 evaluations.
 Range (min … max):  1.603 μs … 338.218 μs  ┊ GC (min … max): 0.00% … 97.51%
 Time  (median):     1.673 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.075 μs ±   4.356 μs  ┊ GC (mean ± σ):  3.14% ±  1.54%

  █▇▅▃▂▁▁▁▂▂▃▂▂▂▂▁▁▁▁ ▁▁   ▁  ▁▁▁▁▁▁▁▁▁▁▁▁▁                   ▂
  ████████████████████████████████████████████▇█▇▇▇▇▇▆▇▆▆▆▆▆▆ █
  1.6 μs       Histogram: log(frequency) by time      6.28 μs <

 Memory estimate: 1.23 KiB, allocs estimate: 10.

In [24]:
function wdot_old(reaction, T, C,s0_R,h_RT)

    @inbounds _kf = @. @view(reaction.Arrhenius_coeffs[:, 1]) * exp(
        @view(reaction.Arrhenius_coeffs[:, 2]) * log(T) -
        @view(reaction.Arrhenius_coeffs[:, 3]) * (4184.0 / R / T),
    )

    for i in reaction.index_three_body
        @inbounds _kf[i] *= dot(@view(reaction.efficiencies_coeffs[:, i]), C)
    end

    for (j, i) in enumerate(reaction.index_falloff)
        @inbounds A0, b0, Ea0 = reaction.Arrhenius_0[j, :]
        @inbounds k0 = A0 * exp(b0 * log(T) - Ea0 * 4184.0 / R / T)
        @inbounds Pr =
            k0 * dot(@view(reaction.efficiencies_coeffs[:, i]), C) / _kf[i]
        lPr = log10(Pr)
        _kf[i] *= (Pr / (1 + Pr))

    #     # reference:
    #     # http://web.mit.edu/2.62/cantera/doc/html/classCantera_1_1Troe4.html#a38aa787421d426dfd0a587fd6fc8108e
        if reaction.index_falloff_Troe[j] > 0
            k = reaction.index_falloff_Troe[j]
            @inbounds F_cent =
                (1 - reaction.Troe_[k, 1]) * exp(-T / reaction.Troe_[k, 4]) +
                reaction.Troe_[k, 1] * exp(-T / reaction.Troe_[k, 2]) +
                exp(-reaction.Troe_[k, 3] / T)

            lF_cent = log10(F_cent)
            _C = -0.4 - 0.67 * lF_cent
            N = 0.75 - 1.27 * lF_cent
            @inbounds f1 = (lPr + _C) / (N - 0.14 * (lPr + _C))
            @inbounds _kf[i] *= exp(log(10.0) * lF_cent / (1 + f1^2))
        end
    end

    @inbounds ΔS_R = reaction.vk' * s0_R
    @inbounds ΔH_RT = reaction.vk' * h_RT
    @inbounds Keq =
        @. exp(ΔS_R - ΔH_RT + log(one_atm / R / T) * reaction.vk_sum)
    @inbounds _kr = @. _kf / Keq * reaction.is_reversible

    for i = 1:reaction.n_reactions
        @inbounds for j in reaction.i_reactant[i]
            @inbounds _kf[i] *= C[j]^reaction.reactant_orders[j, i]
        end
        if reaction.is_reversible[i]
            @inbounds for j in reaction.i_product[i]
                @inbounds _kr[i] *= C[j]^reaction.product_stoich_coeffs[j, i]
            end
            
        end
    end

    #if get_qdot
    #    return _kf - _kr
    #end
    return reaction.vk * (_kf - _kr) #omega_dot_k
end


wdot_old (generic function with 1 method)

In [25]:
wdot_func!(wdot,unified_state.kf,unified_state.kr,unified_state.C,unified_state.g_RT,unifiedReactions,T)
maximum(abs.(wdot - wdot_old(gas.kinetics, T, C,s0_R,h_RT)))

1.0550138540565968e-10